# Plant Disease Recognition: Filling the Research Gaps of an IEEE Base Paper

**Base paper:** S. M. Hassan and A. K. Maji, "Plant Disease Identification Using a Novel Convolutional Neural Network," *IEEE Access*, vol. 10, pp. 5390–5401, 2022. doi:10.1109/ACCESS.2022.3141371

The base paper proposes a lightweight CNN (Inception blocks + residual connections + depthwise separable convolutions, 428,100 parameters). It reports 99.39% on PlantVillage, using a random 80:20 split.

**Gaps filled in this project**

| Gap in the base paper | How this project fills it | Section |
|---|---|---|
| **Gap 1.** Random image-level 80:20 split; the same 20% is used for both validation and testing. Copies of one leaf can be in both train and test. | We **audit** the dataset for leaked images, then split **by leaf** into 70% train / 15% validation / 15% test. The test set is used only once. | 2, 4 |
| **Gap 3.** Tested only on images from the same dataset it was trained on. No lab → field test. | We test every model on **PlantDoc field photos**. We then improve field accuracy by **mixing field images into training** (as Ahmad et al., *IEEE Access* 2023 suggest). | 3, 5, 6 |
| **Gap 6.** No explanation of predictions. | We add **Grad-CAM** heat-maps that show which part of the leaf drove each prediction. | 7 |

**Runtime → Change runtime type → T4 GPU**, then run all cells (about 1.5–2 hours in total). Set `QUICK = True` for a 10-minute test run.

**If Colab disconnects:** just reconnect and click **Run all** again. Trained models and results are saved to your Google Drive (`MyDrive/plant_project/results`). Finished experiments are **skipped**, and a half-trained experiment **continues** from its last saved checkpoint.

In [ ]:
# ===== Main switches =====
QUICK = False          # True = tiny subset, 1 epoch, just to check everything runs (about 10 min)
USE_DRIVE = True       # save models + results to Google Drive so a disconnect does not lose them

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/plant_project/" + ("results_quick" if QUICK else "results")
    os.makedirs(SAVE_DIR, exist_ok=True)
    os.chdir("/content")
    if os.path.islink("results"): os.unlink("results")
    elif os.path.isdir("results"): os.system(f"cp -rn results/. '{SAVE_DIR}/' && rm -rf results")
    os.symlink(SAVE_DIR, "results")
    print("Saving to", SAVE_DIR, "| already saved:", sorted(os.listdir("results")))

In [ ]:
# ===== Settings =====
IMG = 224              # input size
BATCH = 64
SEED = 42
EPOCHS_BASE = 10       # base-paper network (paper used 50; 10 is enough to show the effect)
EPOCHS_HEAD = 1        # MobileNetV3: frozen-backbone epochs
EPOCHS_FT = 3          # MobileNetV3: fine-tuning epochs (pretrained models converge fast)
RESUME_EPOCHS = 3      # extra epochs given to a model that was interrupted half-way
if QUICK:
    EPOCHS_BASE, EPOCHS_HEAD, EPOCHS_FT = 1, 1, 1

import os, re, json, random, shutil
CACHE_DIR = "/content/tfcache" if os.path.isdir("/content") else "tfcache"
shutil.rmtree(CACHE_DIR, ignore_errors=True); os.makedirs(CACHE_DIR, exist_ok=True)   # fresh image cache each run
import numpy as np, pandas as pd
import tensorflow as tf, keras
from keras import layers
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, classification_report
keras.utils.set_random_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE
os.makedirs("results", exist_ok=True)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 1. Download the data
* **New Plant Diseases Dataset**: augmented PlantVillage, 38 classes, about 87.9k images. Original: [Kaggle](https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset). We download a public mirror so no Kaggle login is needed.
* **PlantDoc**: real **field** photos of leaves (Singh et al., 2020), used for the lab → field test.

In [ ]:
if not os.path.exists("npd"):
    !wget -q -O npd.zip "https://huggingface.co/datasets/Sadashiv/Plant-Diseases-Dataset/resolve/main/Plant-Diseases-Dataset.zip"
    !mkdir -p npd && unzip -q npd.zip -d npd && rm npd.zip
if not os.path.exists("PlantDoc-Dataset"):
    !git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset

# find the folder that contains train/ and val(id)/
NPD = next(dp for dp, ds, _ in os.walk("npd") if "train" in ds and ({"val", "valid"} & set(ds)))
VAL_DIR = "valid" if os.path.isdir(f"{NPD}/valid") else "val"
CLASS_NAMES = sorted(os.listdir(f"{NPD}/train"))
NUM_CLASSES = len(CLASS_NAMES)
print(NPD, VAL_DIR, NUM_CLASSES, "classes")

## 2. Gap 1: is there leakage in the usual train/valid split?
The dataset was made by rotating and flipping PlantVillage photos. Each augmented file keeps the original file name and adds a suffix such as `_flipLR`, `_90deg` or `_new30degFlipLR`. If we remove the suffix we get the **leaf ID**. Then we can count how many validation images come from a leaf that is also in the training folder.

In [ ]:
AUG = re.compile(r"_(flipLR|flipTB|90deg|180deg|270deg|new30degFlipLR|newGRR)$", re.I)
def leaf_id(path):
    return AUG.sub("", os.path.splitext(os.path.basename(path))[0]).lower().strip()

rows = []
for split in ("train", VAL_DIR):
    for c in CLASS_NAMES:
        for f in sorted(os.listdir(f"{NPD}/{split}/{c}")):
            p = f"{NPD}/{split}/{c}/{f}"
            rows.append(dict(path=p, cls=c, label=CLASS_NAMES.index(c),
                             orig_split="train" if split == "train" else "valid",
                             leaf=c + "/" + leaf_id(p)))
df = pd.DataFrame(rows)

train_leaves = set(df[df.orig_split == "train"].leaf)
df["leaked"] = (df.orig_split == "valid") & df.leaf.isin(train_leaves)
v = df[df.orig_split == "valid"]
audit = {
    "train_images": int((df.orig_split == "train").sum()),
    "valid_images": int(len(v)),
    "valid_images_leaked": int(v.leaked.sum()),
    "pct_valid_images_leaked": round(100 * v.leaked.mean(), 2),
    "unique_leaves_total": int(df.leaf.nunique()),
}
print(json.dumps(audit, indent=2))
json.dump(audit, open("results/leakage_audit.json", "w"), indent=2)
(v.groupby("cls").leaked.mean() * 100).sort_values().plot.barh(figsize=(7, 9), color="#2f7d4f")
plt.xlabel("% of validation images whose leaf is also in train"); plt.title("Leakage per class"); plt.tight_layout()
plt.savefig("results/leakage_per_class.png", dpi=150); plt.show()

### Fix: split by leaf (70 / 15 / 15)
Every copy of one leaf goes into the **same** split. The validation set is used to choose the best epoch, and the test set is used only once at the end.

In [ ]:
rng = np.random.default_rng(SEED)
assign = {}
for c, g in df.groupby("cls"):
    leaves = np.array(sorted(g.leaf.unique())); rng.shuffle(leaves)
    n = len(leaves); a, b = int(0.70 * n), int(0.85 * n)
    assign.update({l: "train" for l in leaves[:a]})
    assign.update({l: "val" for l in leaves[a:b]})
    assign.update({l: "test" for l in leaves[b:]})
df["split"] = df.leaf.map(assign)
assert df.groupby("leaf").split.nunique().max() == 1      # no leaf in two splits

if QUICK:   # tiny subset for a fast test run
    df = df[df.groupby(["cls", "split", "orig_split"]).cumcount() < 8].reset_index(drop=True)

print(df.groupby("split").agg(images=("path", "size"), leaves=("leaf", "nunique")))
df.to_csv("results/split.csv", index=False)

## 3. Field test set: PlantDoc
PlantDoc folder names are mapped onto our 38 PlantVillage classes (27 test classes and 28 train classes match). The PlantDoc **test** images are only ever used for testing. The PlantDoc **train** images are used in Section 5 to fill Gap 3.

In [ ]:
PLANTDOC_TO_PV = {
 'apple scab leaf': 'Apple___Apple_scab', 'apple leaf': 'Apple___healthy', 'apple rust leaf': 'Apple___Cedar_apple_rust',
 'bell_pepper leaf': 'Pepper,_bell___healthy', 'bell_pepper leaf spot': 'Pepper,_bell___Bacterial_spot',
 'blueberry leaf': 'Blueberry___healthy', 'cherry leaf': 'Cherry_(including_sour)___healthy',
 'corn gray leaf spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'corn leaf blight': 'Corn_(maize)___Northern_Leaf_Blight',
 'corn rust leaf': 'Corn_(maize)___Common_rust_', 'peach leaf': 'Peach___healthy',
 'potato leaf early blight': 'Potato___Early_blight', 'potato leaf late blight': 'Potato___Late_blight',
 'raspberry leaf': 'Raspberry___healthy', 'soyabean leaf': 'Soybean___healthy', 'squash powdery mildew leaf': 'Squash___Powdery_mildew',
 'strawberry leaf': 'Strawberry___healthy', 'tomato early blight leaf': 'Tomato___Early_blight',
 'tomato septoria leaf spot': 'Tomato___Septoria_leaf_spot', 'tomato leaf': 'Tomato___healthy',
 'tomato leaf bacterial spot': 'Tomato___Bacterial_spot', 'tomato leaf late blight': 'Tomato___Late_blight',
 'tomato leaf mosaic virus': 'Tomato___Tomato_mosaic_virus', 'tomato leaf yellow virus': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
 'tomato mold leaf': 'Tomato___Leaf_Mold', 'tomato two spotted spider mites leaf': 'Tomato___Spider_mites Two-spotted_spider_mite',
 'grape leaf': 'Grape___healthy', 'grape leaf black rot': 'Grape___Black_rot'}

def load_plantdoc(split):
    # re-save every image as a clean RGB JPEG so TensorFlow can always decode it
    out, rows = f"plantdoc_clean/{split}", []
    for folder in sorted(os.listdir(f"PlantDoc-Dataset/{split}")):
        pv = PLANTDOC_TO_PV.get(folder.lower().strip())
        if pv is None or pv not in CLASS_NAMES: continue
        os.makedirs(f"{out}/{pv}", exist_ok=True)
        for i, f in enumerate(sorted(os.listdir(f"PlantDoc-Dataset/{split}/{folder}"))):
            dst = f"{out}/{pv}/{i:04d}.jpg"
            try:
                if not os.path.exists(dst):
                    Image.open(f"PlantDoc-Dataset/{split}/{folder}/{f}").convert("RGB").save(dst, quality=95)
                rows.append(dict(path=dst, label=CLASS_NAMES.index(pv)))
            except Exception as e:
                print("skip", f, e)
    return pd.DataFrame(rows)

pd_train, pd_test = load_plantdoc("train"), load_plantdoc("test")
print("PlantDoc train:", len(pd_train), "| test:", len(pd_test), "images in", pd_test.label.nunique(), "classes")

## 4. Data pipeline and the two models

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2, value_range=(0, 255)),
], name="augment")

def load_img(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG, IMG))          # float32 in [0, 255]
    return img, label

def _load_u8(path, label):                          # uint8 version, 4x smaller on disk
    img, label = load_img(path, label)
    return tf.cast(tf.clip_by_value(img, 0, 255), tf.uint8), label

def _to_float(img, label):
    return tf.cast(img, tf.float32), label

# Speed-up: decoded images are cached on Colab's local disk after the first epoch,
# so later epochs (and later experiments) skip JPEG decoding, the main bottleneck.
# The dataset already contains rotated/flipped copies, so only cheap augmentation is used.
def _source(frame, cache_name, shuffle):
    if shuffle: frame = frame.sample(frac=1, random_state=SEED)      # fixed random order
    ds = tf.data.Dataset.from_tensor_slices((frame.path.values, frame.label.values.astype("int32")))
    ds = ds.map(_load_u8, num_parallel_calls=AUTOTUNE)
    if cache_name: ds = ds.cache(f"{CACHE_DIR}/{cache_name}")
    return ds

def make_ds(frame, training=False, cache_name=None):
    ds = _source(frame, cache_name, shuffle=training)
    if training: ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH).map(_to_float, num_parallel_calls=AUTOTUNE)
    if training: ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

def mixed_ds(lab_frame, field_frame, field_share=0.15):
    # Gap 3 fix: each training batch contains ~15% field photos
    lab = _source(lab_frame, "tr", shuffle=True).shuffle(2048, seed=SEED).repeat()
    fld = _source(field_frame, None, shuffle=True).shuffle(1024, seed=SEED).repeat()
    ds = tf.data.Dataset.sample_from_datasets([lab, fld], [1 - field_share, field_share], seed=SEED)
    ds = ds.batch(BATCH).map(_to_float, num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

tr, va, te = (df[df.split == s] for s in ("train", "val", "test"))
orig_tr, orig_va = df[df.orig_split == "train"], df[df.orig_split == "valid"]
print(len(tr), len(va), len(te))

### Model A: re-implementation of the base-paper network
This follows Section III-D and Fig. 5 of Hassan & Maji (2022):
* 1 standard convolution, 3 depthwise separable convolutions and 2 max-pools
* 3 modified Inception-A blocks with residual connections, then a modified Reduction-A block
* 3 modified Inception-B blocks with residual connections, then a modified Reduction-B block
* global average pooling, dropout and a dense layer

The paper does not list filter widths, so they were chosen to match its size (about 430k parameters vs. 428,100 reported).

In [ ]:
def cbr(x, f, k=1, s=1):
    x = layers.Conv2D(f, k, strides=s, padding="same", use_bias=False)(x)
    return layers.ReLU()(layers.BatchNormalization()(x))

def ds_conv(x, f, k=3, s=1):
    x = layers.SeparableConv2D(f, k, strides=s, padding="same", use_bias=False)(x)
    return layers.ReLU()(layers.BatchNormalization()(x))

def inception(x, w, k):            # k=3 -> Inception-A, k=7 -> Inception-B
    c = x.shape[-1]
    b1 = cbr(x, w)
    b2 = ds_conv(cbr(x, w), w, k)
    b3 = ds_conv(ds_conv(cbr(x, w), w, k), w, k)
    b4 = cbr(layers.MaxPool2D(3, 1, "same")(x), w)
    y = layers.BatchNormalization()(layers.Conv2D(c, 1, use_bias=False)(layers.Concatenate()([b1, b2, b3, b4])))
    return layers.ReLU()(layers.Add()([x, y]))   # residual connection

def reduction(x, f):
    return layers.Concatenate()([layers.MaxPool2D(3, 2, "same")(x), ds_conv(cbr(x, f), f, 3, 2)])

def base_paper_net():
    inp = keras.Input((IMG, IMG, 3)); x = layers.Rescaling(1 / 255.)(inp)
    x = cbr(x, 32, 3, 2); x = ds_conv(x, 48); x = layers.MaxPool2D(2)(x)
    x = ds_conv(x, 64); x = ds_conv(x, 96); x = layers.MaxPool2D(2)(x)
    for _ in range(3): x = inception(x, 30, 3)
    x = reduction(x, 96)
    for _ in range(3): x = inception(x, 46, 7)
    x = reduction(x, 128)
    x = layers.Dropout(0.3)(layers.GlobalAveragePooling2D()(x))
    return keras.Model(inp, layers.Dense(NUM_CLASSES, name="logits")(x), name="base_paper_net")

print("Base-paper net parameters:", f"{base_paper_net().count_params():,}")

### Model B: proposed lightweight transfer-learning model
We use MobileNetV3-Large, pretrained on ImageNet and built for phones. It has two training stages: first the new head only, then fine-tuning with BatchNorm frozen. We add data augmentation and label smoothing.

In [ ]:
def proposed_net():
    base = keras.applications.MobileNetV3Large(input_shape=(IMG, IMG, 3), include_top=False,
                                               weights="imagenet", include_preprocessing=True)
    x = layers.Dropout(0.2)(layers.GlobalAveragePooling2D()(base.output))
    m = keras.Model(base.input, layers.Dense(NUM_CLASSES, name="logits")(x), name="proposed_mobilenetv3")
    m.backbone_layers = base.layers
    return m

LOSS = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
def smooth_loss(y, logits, eps=0.1):   # label smoothing for integer labels
    y1 = tf.one_hot(tf.cast(y, tf.int32), NUM_CLASSES) * (1 - eps) + eps / NUM_CLASSES
    return keras.losses.categorical_crossentropy(y1, logits, from_logits=True)

def fit(model, train_ds, val_ds, epochs, lr, name, loss=LOSS, steps=None, best=None):
    ck = f"results/{name}.keras"
    model.compile(optimizer=keras.optimizers.Adam(lr), loss=loss, metrics=["accuracy"])
    h = model.fit(train_ds, validation_data=val_ds, epochs=epochs, steps_per_epoch=steps,
                  callbacks=[keras.callbacks.ModelCheckpoint(ck, monitor="val_accuracy", save_best_only=True,
                                                             initial_value_threshold=best)])
    return keras.models.load_model(ck, compile=False), h.history

def train_proposed(train_ds, name, steps=None):
    m = proposed_net()
    for l in m.backbone_layers: l.trainable = False
    m, h1 = fit(m, train_ds, make_ds(va, cache_name="va"), EPOCHS_HEAD, 1e-3, name, smooth_loss, steps)
    for l in m.layers: l.trainable = not isinstance(l, layers.BatchNormalization)
    m, h2 = fit(m, train_ds, make_ds(va, cache_name="va"), EPOCHS_FT, 1e-4, name, smooth_loss, steps)
    return m, {k: h1[k] + h2[k] for k in h1}

def predict(model, frame):
    return np.concatenate([model.predict_on_batch(x) for x, _ in make_ds(frame)])

def evaluate(model, frame):
    p = predict(model, frame).argmax(1); y = frame.label.values
    return float(round(100 * accuracy_score(y, p), 2)), float(round(100 * f1_score(y, p, labels=sorted(set(y)), average="macro"), 2))

# ---- resume support: skip finished experiments, continue interrupted ones ----
RES_FILE = "results/results.json"
RESULTS = json.load(open(RES_FILE)) if os.path.exists(RES_FILE) else {}
def save_results():
    json.dump(RESULTS, open(RES_FILE, "w"), indent=2, default=float)

def run(name, train_fn, val_frame, lr_resume, loss=LOSS, train_ds=None, steps=None):
    ck, flag = f"results/{name}.keras", f"results/{name}.done"
    if os.path.exists(flag):
        print(f"[{name}] already finished -> loading the saved model (no training)")
        return keras.models.load_model(ck, compile=False)
    if os.path.exists(ck):
        m = keras.models.load_model(ck, compile=False)
        best = evaluate(m, val_frame)[0] / 100
        print(f"[{name}] interrupted earlier (val acc of saved model {best:.4f}) -> {RESUME_EPOCHS} more epochs")
        for l in m.layers: l.trainable = not isinstance(l, layers.BatchNormalization)
        m, _ = fit(m, train_ds, make_ds(val_frame, cache_name="va" if val_frame is va else None), RESUME_EPOCHS, lr_resume, name, loss, steps, best)
    else:
        print(f"[{name}] training from scratch")
        m, _ = train_fn()
    open(flag, "w").write("done")
    return m

## 5. Experiments
| # | Model | Training data | Purpose |
|---|---|---|---|
| E1 | Base-paper net | **original** train/valid folders (the usual way) | Reproduces the usual protocol and **measures the leakage effect** (Gap 1) |
| E2 | Base-paper net | **leaf-grouped** split | Honest accuracy of the base paper's model, plus the field test (Gap 3) |
| E3 | Proposed | leaf-grouped split (lab images only) | Effect of the better model |
| E4 | Proposed + field mixing | leaf-grouped split + PlantDoc train (15% of each batch) | **Fills Gap 3** |

In [ ]:
# ---- E1: base-paper net, usual (leaky) split ----
N1 = "E1 Base-paper net, usual split"
m1 = run("E1_base_orig_split",
         lambda: fit(base_paper_net(), make_ds(orig_tr, True), make_ds(orig_va), EPOCHS_BASE, 1e-3, "E1_base_orig_split"),
         orig_va, 3e-4, train_ds=make_ds(orig_tr, True))
if N1 not in RESULTS:
  p = predict(m1, orig_va).argmax(1); ok = p == orig_va.label.values; lk = orig_va.leaked.values
  RESULTS[N1] = {
    "valid acc (all)": float(round(100 * ok.mean(), 2)),
    "valid acc: leaked leaves": float(round(100 * ok[lk].mean(), 2)) if lk.any() else None,
    "valid acc: unseen leaves": float(round(100 * ok[~lk].mean(), 2)) if (~lk).any() else None,
    "PlantDoc field acc": evaluate(m1, pd_test)[0]}
  save_results()
print(RESULTS[N1])
del m1; import gc; gc.collect()       # free memory (only m2 and m4 are needed later)

In [ ]:
# ---- E2: base-paper net, leakage-free split ----
N2 = "E2 Base-paper net, leaf split"
m2 = run("E2_base_leaf_split",
         lambda: fit(base_paper_net(), make_ds(tr, True, "tr"), make_ds(va, cache_name="va"), EPOCHS_BASE, 1e-3, "E2_base_leaf_split"),
         va, 3e-4, train_ds=make_ds(tr, True, "tr"))
if N2 not in RESULTS:
    acc, f1 = evaluate(m2, te)
    RESULTS[N2] = {"test acc": acc, "test macro-F1": f1, "PlantDoc field acc": evaluate(m2, pd_test)[0], "params": m2.count_params()}
    save_results()
print(RESULTS[N2])

In [ ]:
# ---- E3: proposed, lab images only ----
N3 = "E3 Proposed (lab only)"
m3 = run("E3_proposed_lab", lambda: train_proposed(make_ds(tr, True, "tr"), "E3_proposed_lab"),
         va, 1e-4, smooth_loss, train_ds=make_ds(tr, True, "tr"))
if N3 not in RESULTS:
    acc, f1 = evaluate(m3, te)
    RESULTS[N3] = {"test acc": acc, "test macro-F1": f1, "PlantDoc field acc": evaluate(m3, pd_test)[0], "params": m3.count_params()}
    save_results()
print(RESULTS[N3])
del m3; gc.collect()

In [ ]:
# ---- E4: proposed + field-image mixing (fills Gap 3) ----
steps = int(np.ceil(len(tr) / BATCH))
N4 = "E4 Proposed + field mixing"
m4 = run("E4_proposed_field_mix", lambda: train_proposed(mixed_ds(tr, pd_train), "E4_proposed_field_mix", steps=steps),
         va, 1e-4, smooth_loss, train_ds=mixed_ds(tr, pd_train), steps=steps)
if N4 not in RESULTS:
    acc, f1 = evaluate(m4, te)
    RESULTS[N4] = {"test acc": acc, "test macro-F1": f1, "PlantDoc field acc": evaluate(m4, pd_test)[0], "params": m4.count_params()}
    save_results()
print(RESULTS[N4])

## 6. Results

In [ ]:
res = pd.DataFrame(RESULTS).T
res.to_csv("results/results_table.csv"); save_results()
display(res)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
e1 = RESULTS["E1 Base-paper net, usual split"]
ax[0].bar(["all valid", "leaked leaves", "unseen leaves"],
          [e1["valid acc (all)"], e1["valid acc: leaked leaves"] or 0, e1["valid acc: unseen leaves"] or 0],
          color=["#9bbfa8", "#c0504d", "#2f7d4f"])
ax[0].set_title("Gap 1: accuracy is higher on leaked images"); ax[0].set_ylabel("accuracy (%)")
names = ["E2 Base-paper net, leaf split", "E3 Proposed (lab only)", "E4 Proposed + field mixing"]
x = np.arange(3)
ax[1].bar(x - 0.2, [RESULTS[n]["test acc"] for n in names], 0.4, label="lab test (leaf split)", color="#9bbfa8")
ax[1].bar(x + 0.2, [RESULTS[n]["PlantDoc field acc"] for n in names], 0.4, label="PlantDoc field photos", color="#2f7d4f")
ax[1].set_xticks(x, ["Base-paper net", "Proposed", "Proposed + field"]); ax[1].legend(); ax[1].set_title("Gap 3: lab vs field accuracy")
plt.tight_layout(); plt.savefig("results/results.png", dpi=150); plt.show()

## 7. Gap 6: Grad-CAM explanations
Grad-CAM (Selvaraju et al., ICCV 2017) highlights the image region that most increased the predicted class score. Red means important. A trustworthy model should light up the **lesions on the leaf**, not the background.

In [ ]:
def last_4d_layer(model):
    for l in reversed(model.layers):
        if len(l.output.shape) == 4: return l.name

def gradcam(model, img):
    gm = keras.Model(model.inputs, [model.get_layer(last_4d_layer(model)).output, model.outputs[0]])
    x = tf.convert_to_tensor(img[None], tf.float32)
    with tf.GradientTape() as tape:
        conv, logits = gm(x, training=False)
        c = int(tf.argmax(logits[0])); score = logits[:, c]
    g = tape.gradient(score, conv)
    cam = tf.nn.relu(tf.reduce_sum(tf.reduce_mean(g, axis=(1, 2), keepdims=True) * conv, -1))[0]
    cam = tf.image.resize(cam[..., None], (IMG, IMG))[..., 0].numpy()
    return cam / (cam.max() + 1e-8), c

def show_gradcam(models, frame, n=4, title="", fname="gradcam.png"):
    sample = frame.sample(n, random_state=1)
    fig, axs = plt.subplots(n, len(models) + 1, figsize=(3 * (len(models) + 1), 3 * n))
    for i, (p, y) in enumerate(zip(sample.path, sample.label)):
        img = load_img(p, y)[0].numpy()
        axs[i, 0].imshow(img.astype("uint8")); axs[i, 0].set_title("true: " + CLASS_NAMES[y].split("___")[-1][:22], fontsize=8)
        for j, (name, m) in enumerate(models.items(), 1):
            cam, c = gradcam(m, img)
            axs[i, j].imshow(img.astype("uint8")); axs[i, j].imshow(cam, cmap="jet", alpha=0.45)
            axs[i, j].set_title(f"{name}\npred: {CLASS_NAMES[c].split('___')[-1][:20]}", fontsize=8,
                                color="green" if c == y else "red")
    for a in axs.ravel(): a.axis("off")
    plt.suptitle(title); plt.tight_layout(); plt.savefig(f"results/{fname}", dpi=150); plt.show()

show_gradcam({"Base-paper net": m2, "Proposed + field": m4}, te, title="Lab test images", fname="gradcam_lab.png")
show_gradcam({"Base-paper net": m2, "Proposed + field": m4}, pd_test, title="PlantDoc field photos", fname="gradcam_field.png")

## 8. Save the final model for the app

In [ ]:
os.makedirs("app_model", exist_ok=True)
m4.save("app_model/model.keras")
json.dump({"class_names": CLASS_NAMES, "img_size": IMG}, open("app_model/meta.json", "w"))
!zip -qr results.zip results/ app_model
print("Download results.zip from the Files panel (left). It has the model, the tables and the figures.")
# from google.colab import files; files.download("results.zip")

## What to write in your report

**Base paper.** Hassan & Maji (IEEE Access, 2022) proposed a lightweight Inception–residual CNN with 428,100 parameters and reported 99.39% accuracy on PlantVillage using a random 80:20 split.

**Gaps addressed in this project**
1. **Evaluation leakage (Gap 1).** Random image-level splits let augmented copies of the same leaf appear in both training and testing. Our audit found that *(see `results/leakage_audit.json`)* % of the validation images in the standard split were copies of training leaves. We therefore split the data by leaf (70/15/15) and report accuracy on a held-out test set that is used only once.
2. **Lab-to-field generalisation (Gap 3).** The base paper never tests a lab-trained model on field photographs. We test all models on PlantDoc field images and improve field accuracy by mixing field images into training (following Ahmad et al., *IEEE Access*, 2023).
3. **Explainability (Gap 6).** The base paper gives no explanation of its predictions. We add Grad-CAM heat-maps that show which leaf regions drive each decision, both in the notebook and in the app.

Fill in the numbers from `results/results_table.csv`.